# 12 — Week 4: Chronos-Bolt (small) zero-shot smoke test and timing

Zero-shot = no training on our data. Point forecast = median (0.5 quantile), which is the MAE-optimal choice.
Contexts: 60 minutes (our protocol window) and 512 minutes (longer context; invalid minutes passed as NaN, which
Chronos accepts). Checked on 2,000 validation origins only: speed on CPU vs MPS and MAE vs persistence.
Weights are read from the local Hugging Face cache (offline). **No test data is touched.**
Leakage caveat (Week 1 lineage check): MIT Supercloud is not in the documented pre-training corpora; results are
secondary.

In [1]:
# Cell 1: setup (offline: weights already in the cache)
import os
os.environ["HF_HUB_OFFLINE"] = "1"
import sys, json, time, warnings, traceback
from pathlib import Path
import numpy as np
import polars as pl
import torch

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
from models import rnn as R
from chronos import BaseChronosPipeline
import chronos, transformers

MODEL_ID = "amazon/chronos-bolt-small"
QL = [0.1, 0.5, 0.9]
H_MAX = max(R.HORIZONS)
OUT = ROOT / "reports" / "week4"
OUT.mkdir(parents=True, exist_ok=True)
RES = {"model": MODEL_ID, "chronos": getattr(chronos, "__version__", "?"), "transformers": transformers.__version__,
       "torch": torch.__version__, "mps": bool(torch.backends.mps.is_available())}
DEVICES = ["cpu"] + (["mps"] if RES["mps"] else [])
pipes = {}
for dev in DEVICES:
    try:
        t = time.time()
        pipes[dev] = BaseChronosPipeline.from_pretrained(MODEL_ID, device_map=dev, torch_dtype=torch.float32)
        print(f"loaded on {dev} in {time.time() - t:.1f} s")
    except Exception as e:  # noqa: BLE001
        print(f"could not load on {dev}: {type(e).__name__}: {e}")
n_params = sum(p.numel() for p in pipes["cpu"].model.parameters())
RES["n_params"] = int(n_params)
print(json.dumps(RES, indent=1), "| params", f"{n_params:,}")


def forecast(pipe, ctx, batch=256):
    """ctx: (N, L) float32 with NaN for missing minutes -> median forecasts (N, 15) and quantiles (N, 15, 3)."""
    meds, qs = [], []
    for i in range(0, len(ctx), batch):
        q, _ = pipe.predict_quantiles(torch.from_numpy(ctx[i:i + batch]), prediction_length=H_MAX, quantile_levels=QL)
        q = q.float().cpu().numpy()
        qs.append(q)
        meds.append(q[:, :, QL.index(0.5)])
    return np.concatenate(meds), np.concatenate(qs)


def contexts(Pnan, t, L):
    idx = t[:, None] + np.arange(-(L - 1), 1)[None, :]
    c = np.full(idx.shape, np.nan, dtype=np.float32)
    ok = idx >= 0
    c[ok] = Pnan[idx[ok]]
    return c


Loading weights: 100%|█████████████████████| 143/143 [00:00<00:00, 6364.24it/s]


loaded on cpu in 0.1 s


Loading weights: 100%|████████████████████| 143/143 [00:00<00:00, 41823.13it/s]


loaded on mps in 0.7 s
{
 "model": "amazon/chronos-bolt-small",
 "chronos": "2.3.2",
 "transformers": "5.19.0",
 "torch": "2.14.1",
 "mps": true,
 "n_params": 47718016
} | params 47,718,016


In [2]:
# Cell 2: SYNTHETIC check (shapes, NaN handling, sensible accuracy on a smooth series)
rng = np.random.default_rng(0)
n = 5000
tt = np.arange(n)
Psyn = (30 + 3 * np.sin(2 * np.pi * tt / 240) + rng.normal(0, 0.2, n)).astype(np.float32)
Psyn[1000:1100] = np.nan
orig = np.arange(1200, 4900, 37)
syn = {}
for L in (60, 512):
    c = contexts(Psyn, orig, L)
    med, q = forecast(pipes["cpu"], c)
    ok = np.isfinite(med).all() and med.shape == (len(orig), H_MAX) and q.shape == (len(orig), H_MAX, len(QL))
    row = {"L": L, "shape_ok": bool(ok), "nan_in_context_rows": int(np.isnan(c).any(axis=1).sum())}
    for h in R.HORIZONS:
        y = Psyn[orig + h]
        row[f"skill_h{h}"] = round(1 - np.mean(np.abs(med[:, h - 1] - y)) / np.mean(np.abs(Psyn[orig] - y)), 3)
    row["quantiles_ordered"] = bool((np.diff(q, axis=2) >= -1e-6).all())
    syn[L] = row
    print(row)
RES["synthetic"] = syn
good = all(r["shape_ok"] and r["quantiles_ordered"] and r["skill_h15"] > 0 for r in syn.values())
print("\nSYNTHETIC CHECK PASSED" if good else "\nSYNTHETIC CHECK NOT PASSED: send this output")


{'L': 60, 'shape_ok': True, 'nan_in_context_rows': 0, 'skill_h1': np.float32(0.116), 'skill_h5': np.float32(0.125), 'skill_h15': np.float32(0.298), 'quantiles_ordered': True}
{'L': 512, 'shape_ok': True, 'nan_in_context_rows': 12, 'skill_h1': np.float32(0.085), 'skill_h5': np.float32(0.117), 'skill_h15': np.float32(0.44), 'quantiles_ordered': True}

SYNTHETIC CHECK PASSED


In [3]:
# Cell 3: REAL data, 2,000 validation origins: timing (CPU vs MPS) and accuracy vs persistence
md = pl.read_parquet(ROOT / "data" / "processed" / "model_dataset.parquet")
data = R.Data(md, {"scaling": "local", "calendar": False, "covariates": False})
valid = md["valid"].to_numpy()
Pnan = np.where(valid, md["P_kW"].fill_null(np.nan).to_numpy(), np.nan).astype(np.float32)
va_all = data.origins("val")
rng = np.random.default_rng(0)
va = np.sort(rng.choice(va_all, 2000, replace=False))
rows = []
for L in (60, 512):
    c = contexts(Pnan, va, L)
    for dev, pipe in pipes.items():
        try:
            forecast(pipe, c[:64])                                   # warm-up
            t = time.time()
            med, q = forecast(pipe, c)
            s = time.time() - t
            row = {"L": L, "device": dev, "ms_per_origin": round(1000 * s / len(va), 3),
                   "est_min_full_val": round(s / len(va) * len(va_all) / 60, 1),
                   "nan_share_context": round(float(np.isnan(c).mean()), 4)}
            for j, h in enumerate(R.HORIZONS):
                sel = np.isin(va, data.idx[h]["val"])
                y, p0 = data.P[va[sel] + h], data.P[va[sel]]
                m = float(np.mean(np.abs(med[sel, h - 1] - y)))
                row[f"MAE_h{h}"] = round(m, 4)
                row[f"skill_h{h}"] = round(1 - m / float(np.mean(np.abs(p0 - y))), 3)
        except Exception as e:  # noqa: BLE001
            row = {"L": L, "device": dev, "error": f"{type(e).__name__}: {str(e)[:200]}"}
            traceback.print_exc(limit=3)
        rows.append(row)
        print(row)
RES["real_2000"] = rows
RES["n_val_origins"] = int(len(va_all))


{'L': 60, 'device': 'cpu', 'ms_per_origin': 0.681, 'est_min_full_val': 0.5, 'nan_share_context': 0.0, 'MAE_h1': 0.2721, 'skill_h1': 0.019, 'MAE_h5': 0.438, 'skill_h5': -0.001, 'MAE_h15': 0.6363, 'skill_h15': -0.082}
{'L': 60, 'device': 'mps', 'ms_per_origin': 0.646, 'est_min_full_val': 0.5, 'nan_share_context': 0.0, 'MAE_h1': 0.2721, 'skill_h1': 0.019, 'MAE_h5': 0.438, 'skill_h5': -0.001, 'MAE_h15': 0.6363, 'skill_h15': -0.082}
{'L': 512, 'device': 'cpu', 'ms_per_origin': 3.05, 'est_min_full_val': 2.2, 'nan_share_context': 0.0054, 'MAE_h1': 0.2776, 'skill_h1': -0.001, 'MAE_h5': 0.4224, 'skill_h5': 0.035, 'MAE_h15': 0.5721, 'skill_h15': 0.027}
{'L': 512, 'device': 'mps', 'ms_per_origin': 1.773, 'est_min_full_val': 1.3, 'nan_share_context': 0.0054, 'MAE_h1': 0.2776, 'skill_h1': -0.001, 'MAE_h5': 0.4224, 'skill_h5': 0.035, 'MAE_h15': 0.5721, 'skill_h15': 0.027}


In [4]:
# Cell 4: save
(OUT / "chronos_smoke.json").write_text(json.dumps(RES, indent=1, default=str))
print("written:", OUT / "chronos_smoke.json")


written: /Users/aryaneuse/Desktop/Uni Project/datacenter-power-forecasting/reports/week4/chronos_smoke.json


In [6]:
# Cell 5: export outputs (save the notebook first with Cmd+S)
%run ../src/tools/nb_outputs_to_md.py 13_week4_chronos_smoke.ipynb


written 13_week4_chronos_smoke_outputs.md | code cells 5 | figures 0
